# A volume as Gaussian splats, with luxar

The worked example is a sunflower head from X-ray micro-CT. Everything here was
run end to end on a laptop with an NVIDIA RTX 500 (4 GB).:

| | |
|---|---|
| published scan | 1371 × 1212 × 1841 uint16, **3.06 G voxels, 6.1 GB** |
| prepared for fitting | 666 × 580 × 844 uint8, 326 M voxels, 3% carrying signal |
| fit | **186,744 splats, well under an hour** |
| web scene | **5.2 MB**, opens in any WebGL2 browser |

**The CUDA kernels have to be compiled**, and **the background has to be exactly zero**.

## 1. Install

The environment is `visualization_software/requirements_luxar.txt`:

```bash
uv venv .venv_luxar --python 3.12 --seed
uv pip install --python .venv_luxar -r visualization_software/requirements_luxar.txt
uv run --python .venv_luxar jupyter lab
```

That covers luxar with the `gsplats` and `io` extras, torch, and ninja.
The CUDA toolchain is not in it - step 2 installs that, because which
version you need depends on the torch you ended up with.


## 2. Compile the CUDA kernels

luxar cannot ship these in the wheel, so a fresh install falls back to plain
PyTorch. The fallback is not a little slower, it is **orders of magnitude**
slower: on the machine above it managed about one iteration per second, against
a few thousand per minute once the kernels were built. If a fit looks broken
rather than slow, this is why.

`nvcc`, `nvvm`, `crt` and `cccl` all have to be the **same CUDA minor version as
your PyTorch**, or the build fails in confusing ways - a newer `nvvm` emits PTX
that an older `ptxas` refuses. Check `torch.version.cuda` and pin to match.

In [4]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

2.14.0+cu130 13.0 True


In [5]:
# 13.0 is what the pinned torch 2.14.0 ships with, so this matches as
# long as you installed from requirements_luxar.txt. If the cell above
# printed a different torch.version.cuda, change all four to match it.
%pip install "nvidia-cuda-nvcc==13.0.*" "nvidia-cuda-cccl==13.0.*" \
             "nvidia-nvvm==13.0.*" "nvidia-cuda-crt==13.0.*"


  Using cached nvidia_cuda_nvcc-13.0.88-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_cccl-13.0.85-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_nvvm-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_crt-13.0.88-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
Using cached nvidia_cuda_nvcc-13.0.88-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (37.4 MB)
Using cached nvidia_cuda_cccl-13.0.85-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)
Using cached nvidia_nvvm-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl (61.6 MB)
Using cached nvidia_cuda_crt-13.0.88-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nvidia-cuda-nvcc]3/4 [nvidia-cuda-nvcc]
Note: you may need to restart the kernel to use updat

In [6]:
import os, subprocess, sys
from pathlib import Path

nv = Path(sys.prefix) / "lib" / f"python3.{sys.version_info.minor}" / "site-packages" / "nvidia" / "cu13"

# The build links against -lcudart, and the wheel only ships libcudart.so.13.
(nv / "lib64").unlink(missing_ok=True)
(nv / "lib64").symlink_to(nv / "lib")
so = nv / "lib" / "libcudart.so"
so.unlink(missing_ok=True)
so.symlink_to("libcudart.so.13")

import luxar
ext = Path(luxar.__file__).parent / "gsplats" / "models" / "gsplats" / "cuda"
env = {**os.environ, "CUDA_HOME": str(nv), "PATH": f"{nv}/bin:{os.environ['PATH']}",
       "CUDA_ARCHS": "89",   # your card's compute capability; 89 = Ada
       "MAX_JOBS": "8"}
print(subprocess.run([sys.executable, "build.py"], cwd=ext, env=env,
                     capture_output=True, text=True).stdout[-600:])

_cpu -ltorch_cuda -ltorch -ltorch_python -L/media/data/Development/hi/repos/workshops/3d-data-visualization/.venv_luxar/lib/python3.12/site-packages/nvidia/cu13/lib64 -lcudart -o cuda_splatting_backend.so

SUCCESS: Built cuda_splatting_backend.so
  Location: /media/data/Development/hi/repos/workshops/3d-data-visualization/.venv_luxar/lib/python3.12/site-packages/luxar/gsplats/models/gsplats/cuda/cuda_splatting_backend.so
  Build info: /media/data/Development/hi/repos/workshops/3d-data-visualization/.venv_luxar/lib/python3.12/site-packages/luxar/gsplats/models/gsplats/cuda/cuda_build_info.json



## 3. Prepare the volume

**Mask the background to exactly zero before fitting.** A fit spends parameters
wherever there is signal, so a background left at a faint non-zero costs splats
and blurs the result.

On the sunflower, windowing the air and the foam holder to zero took the
fraction of voxels carrying signal from 84% down to 3%, and the whole splat
budget went into the flower instead of the tub it sat in.

The cell below runs on the frog head from `example_data/`, which does the same
thing on a volume small enough to fit quickly. `tools/make-sunflower-splats.py`
is the sunflower version, straight out of the published zip files.


In [ ]:
import numpy as np, tifffile, pathlib

pathlib.Path("results/luxar").mkdir(parents=True, exist_ok=True)

# The frog head from example_data, so this runs without downloading anything.
# Point it at your own volume and pick LO/HI off its histogram.
VOLUME = "../example_data/ceratophrys-ornata/frog-head-256x256x195.tif"
LO, HI = 50.0, 220.0       # 8-bit here; the sunflower was uint16 at 20000-60000

vol = tifffile.imread(VOLUME).astype(np.float32)
print(f"raw      {vol.shape}, {vol.size/1e6:.1f} Mvoxel, "
      f"{100*(vol>3).mean():.1f}% above background")

vol = np.clip((vol - LO) / (HI - LO), 0, 1) * 255

# Crop away the margin that is now pure zero, so no tile of the fit is wasted.
nz = np.nonzero(vol > 3)
sl = tuple(slice(int(a.min()), int(a.max()) + 1) for a in nz)
vol = vol[sl].astype(np.uint8)

print(f"prepared {vol.shape}, {vol.size/1e6:.1f} Mvoxel, "
      f"{100*(vol>3).mean():.1f}% carries signal")
tifffile.imwrite("results/luxar/prepared.tif", vol, compression="deflate")


## 4. Fit, build the levels, convert

In [ ]:
%%bash
export CUDA_HOME=$(python -c "import sys,pathlib;print(pathlib.Path(sys.prefix)/'lib'/f'python3.{sys.version_info.minor}'/'site-packages'/'nvidia'/'cu13')")
export PATH=$CUDA_HOME/bin:$PATH
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

luxar gsplat fit results/luxar/prepared.tif results/luxar/fit.gsplats.zarr \
    --preset standard --seeds 200000 -d cuda \
    --tiling uniform --tile-size 192 --overlap 24

luxar gsplat flatten results/luxar/fit.gsplats.zarr results/luxar/flat.gsplats.zarr
luxar gsplat lod     results/luxar/flat.gsplats.zarr results/luxar/scene.gsplats.zarr --recipe levels
luxar gsplat convert results/luxar/scene.gsplats.zarr results/luxar/scene.luxar.zarr --colormap gray --tone-mapping ACES

`lod --recipe` picks the streaming topology. `levels` suits zooming across
scales; `stream` is the fastest first paint for small data; `tiles`, `overview`
and `adaptive` are for progressively larger scenes.

## 5. Look at it

`luxar viewer` serves the bundled viewer and the data together.

In [ ]:
!luxar viewer --data results/luxar/scene.luxar.zarr --port 5173 --data-port 8099

## 6. Hand it to someone else

`luxar export` writes a self-contained folder - viewer, scene and a `serve.py`
that needs nothing but Python 3. That folder is the whole deliverable: unzip and
run, no install.

It is also just static files, so the same folder works on any web host that sets
CORS - see the hosting slide of the session.

In [ ]:
!luxar export results/luxar/scene.luxar.zarr -o results/luxar/live/ --overwrite
# then: cd results/luxar/live && python serve.py

### Sources

- [luxar](https://github.com/royerlab/luxar) · [documentation](https://royerlab.github.io/luxar/) · [88 live demo scenes](https://demos.luxarviewer.dev)
- Sunflower: Chitwood, Quigley & Frank, [*X-ray CT Botanical Images - Set 1*](https://doi.org/10.5281/zenodo.15684909), Zenodo 2025, CC BY 4.0